In [ ]:
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.1 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 26ms
Prepared 2 packages in 443ms
Uninstalled 2 packages in 288ms
Installed 2 packages in 52ms
 - tokenizers==0.22.2
 + tokenizers==0.21.4
 - transformers==4.57.6
 + transformers==4.55.4
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 1ms
Prepared 1 package in 32ms
Uninstalled 1 package in 1ms
Installed 1 package in 6ms
 - trl==0.24.0
 + trl==0.22.2


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from unsloth import FastLanguageModel
from transformers import TextStreamer
import json # 딕셔너리를 예쁘게 출력하기

/tmp/ipython-input-3802121506.py:3: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# 가장 간단한 Agent 형태  
 - Agent용 LLM모델로 "unsloth/Qwen3-14B"사용

## LLM Model class

In [ ]:
class LLMModel:
    def __init__(self, model_name: str = "unsloth/Qwen3-14B", max_seq_length: int = 2048,
                 load_in_4bit: bool = True, load_in_8bit: bool = False, full_finetuning: bool = False,
                 device_map: str = "auto"):
        # Initialize the LLMModel with model and tokenizer loaded internally.
        print("[LLMModel Init] 모델 및 토크나이저 로딩을 시작합니다...")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,   ## 4bit load 사용
            load_in_8bit=load_in_8bit,
            full_finetuning=full_finetuning,
            device_map=device_map,
        )
        print("[LLMModel Init] 모델 및 토크나이저 로딩 완료.")

    def generate_response(self, messages: list[dict], max_new_tokens: int = 1024, temperature: float = 0.3,
                         top_p: float = 0.9, top_k: int = 20) -> str:
        # Generate a response from the LLM based on the input messages.
        print("[LLMModel] 응답 생성 시작...")
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False ,  ## True
        )

        inputs = self.tokenizer(text, return_tensors="pt")
        model_device = next(self.model.parameters()).device
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            streamer=TextStreamer(self.tokenizer, skip_prompt=True),
        )

        # 첫 번째 배치(=질문)에 대한 토큰 시퀀스를 가져와 디코딩
        decoded = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        print("[LLMModel] 응답 생성 완료.")
        return decoded

## Agent class

In [ ]:
class Agent:
    def __init__(self, llm_model: LLMModel):
        # 에이전트가 사용할 수 있는 외부 함수(도구)들을 여기에 등록합니다.
        self.tools = {}

        # TODO: Memory (메모리) 및 상태 관리 초기화
        self.memory = ""

        # 외부에서 주입된 LLMModel 설정
        self.llm_model = llm_model

        print("1. [Agent Init] 에이전트 초기화 완료.")

    def plan(self, query: str) -> list[str]:
        """
        [Planner] 사용자의 쿼리를 분석하고 실행 계획(단계)을 생성합니다.
        본 모듈에서는 키워드 기반의 간단한 라우팅(어떤 툴을 쓸지 결정)을 구현합니다.
        더 발전된 에이전트는 이 부분을 LLM을 이용해 동적으로 계획을 생성합니다.
        Args:     query (str): 사용자의 질문.

        Returns:  list[str]: 실행할 단계들의 리스트.
        """
        print(f"2. [Planner] 쿼리 분석 및 계획 수립 시작: '{query}'")
        # 현재는 매우 단순하게, 받은 쿼리 자체를 하나의 실행 단계로 간주합니다.
        # 과제 파트에서 이 부분을 더 지능적으로 수정할 예정입니다.
        steps = [query]
        print(f"3. [Planner] 계획 수립 완료: {steps}")
        return steps

    def execute(self, steps: list[str]) -> str:
        """
        [Executor] Planner가 생성한 계획에 따라 각 단계를 실행합니다.
        단계가 툴 호출인지, LLM에게 질문하는 것인지 판단하고 실행합니다.
        Args:   steps (list[str]): 실행할 단계들의 리스트.
        Returns:str: 최종 실행 결과.
        """
        print("4. [Executor] 계획 실행 시작...")
        # 현재는 오직 LLM 호출만 존재한다고 가정합니다.
        # steps 리스트의 모든 내용을 하나의 프롬프트로 합칩니다.
        prompt = "\n".join(steps)
        messages = [{"role": "user", "content": prompt}]

        # LLMModel을 통해 응답 생성
        result = self.llm_model.generate_response(messages)
        print("5. [Executor] 실행 완료.")
        return result

## LLMModel 인스턴스 생성

In [ ]:
%%time
# LLMModel 인스턴스 생성
llm_model = LLMModel()
# Wall time: 48.3 s

[LLMModel Init] 모델 및 토크나이저 로딩을 시작합니다...
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

[LLMModel Init] 모델 및 토크나이저 로딩 완료.
CPU times: user 31.9 s, sys: 28.9 s, total: 1min
Wall time: 1min 18s


## Agent 생성 및 실행

In [ ]:
# Agent 인스턴스 생성 (LLMModel 주입)
agent = Agent(llm_model)

# 사용자 쿼리 정의
query = "반도체 8대 공정 중에서 포토리소그래피(Photolithography) 단계에 대해서 자세히 설명해 줘."

# 에이전트 실행
# 계획 수립 (Plan)
steps = agent.plan(query)

# 계획 실행 (Execute)
answer = agent.execute(steps)

# 결과 출력
print("\n\n--- 최종 답변 ---")
print(answer)
# Wall time: 1min 15s

1. [Agent Init] 에이전트 초기화 완료.
2. [Planner] 쿼리 분석 및 계획 수립 시작: '반도체 8대 공정 중에서 포토리소그래피(Photolithography) 단계에 대해서 자세히 설명해 줘.'
3. [Planner] 계획 수립 완료: ['반도체 8대 공정 중에서 포토리소그래피(Photolithography) 단계에 대해서 자세히 설명해 줘.']
4. [Executor] 계획 실행 시작...
[LLMModel] 응답 생성 시작...
반도체 제조 공정 중 **포토리소그래피**(Photolithography)는 반도체 소자의 미세한 회로를 형성하는 데 핵심적인 역할을 하는 공정입니다. 이 공정은 **광학적 기법**을 사용하여 **반도체 기판**(일반적으로 실리콘 웨이퍼)에 **회로 패턴**을 전달하는 과정입니다.

---

## 🔍 1. 포토리소그래피란?

**포토리소그래피**(Photolithography)는 "빛(광)을 이용한 조각 기술"이라는 의미로, **광학 마스크**(Mask)를 사용하여 **웨이퍼 표면에 회로 패턴을 전사**(Transfer)하는 과정입니다.

---

## 🧩 2. 포토리소그래피의 목적

- 반도체 소자의 **미세한 회로**(예: 트랜지스터, 커패시터 등)를 **정확하게 제작**하는 것.
- **패턴 정확도**와 **미세도**(Feature Size)를 제어하여 **고집적도**(High Integration)를 달성하는 것.

---

## 📌 3. 포토리소그래피 공정의 주요 단계

포토리소그래피 공정은 일반적으로 다음과 같은 **5단계**로 구성됩니다:

---

### 🔹 1) **웨이퍼 준비 (Wafer Preparation)**

- **실리콘 웨이퍼**(Silicon Wafer)에 **광결합층**(Photoresist)을 도포합니다.
  - **광결합층**(Photoresist)은 **광감광성**(Light-sensitive) 물질로, 빛에 노출되면 화학적 반응을 일으킵니다.
  - **양자점**(Posi

# Agent 수정 : 정보 조회 추가

### 공정 파라미터를 조회하는 함수

In [ ]:
# 더미(dummy) 공정 파라미터를 조회하는 함수

def get_step_parameters(step_name: str) -> dict:
    """
    주어진 공정 단계(step_name)의 파라미터를 조회하는 더미 함수입니다.
    실제 환경에서는 이 함수 내부에서 데이터베이스에 연결하여 데이터를 조회하는 로직이 들어갑니다.
    Args: step_name (str): 조회할 공정의 이름 (예: "포토리소그래피", "식각").
    Returns: dict: 해당 공정의 파라미터 딕셔너리. 없으면 빈 딕셔너리 반환.
    """
    print(f"--- [Tool Called] get_step_parameters(step_name='{step_name}') ---")

    # 실제 DB 대신 사용할 더미 데이터
    dummy_data = {
        "포토리소그래피": {"노광시간(Exposure Time)": "5.2초", "레진두께(PR Thickness)": "210nm", "초점(Focus)": "+0.1um"},
        "식각": {"가스(Gas)": "CF4 100sccm", "압력(Pressure)": "50mTorr", "전력(RF Power)": "300W"},
        "증착": {"온도(Temperature)": "450°C", "물질(Material)": "Si3N4", "두께(Thickness)": "50Å"}
    }
    # step_name 에 해당하는 키워드가 포함된 키를 찾아 반환
    for key, value in dummy_data.items():
        if key in step_name:
            return value
    return {}

In [ ]:
class Agent:
    def __init__(self, llm_model: LLMModel):
        # 툴 등록: 문자열 이름과 실제 함수를 매핑하는 딕셔너리
        self.tools = {
            "get_step_parameters": get_step_parameters
        }
        print(f"1. [Agent Init] 사용 가능한 툴: {list(self.tools.keys())}")

        # TODO: Memory (메모리) 및 상태 관리 초기화
        self.memory = ""

        # 외부에서 주입된 LLMModel 설정
        self.llm_model = llm_model

        print("1. [Agent Init] 에이전트 초기화 완료.")

    def plan(self, query: str) -> list[str]:
        print(f"2. [Planner] 쿼리 분석 및 계획 수립 시작: '{query}'")
        # 지능적인 계획 수립
        if "파라미터" in query or "정보" in query:
            # 공정 이름(예: '포토리소그래피')을 쿼리에서 추출
            step_name = query.split(" ")[0]  # 간단히 첫 단어를 공정 이름으로 가정
            plan = [f"TOOL_CALL:get_step_parameters({step_name})"]
        else:
            plan = [f"LLM_CALL:{query}"]

        print(f"3. [Planner] 계획 수립 완료: {plan}")
        return plan

    def execute(self, steps: list[str]) -> str:
        print("4. [Executor] 계획 실행 시작...")
        final_answer = ""
        for step in steps:
            if step.startswith("TOOL_CALL:"):
                # 툴 호출 명령 파싱
                tool_call_str = step.replace("TOOL_CALL:", "")
                tool_name = tool_call_str.split("(")[0]
                tool_arg = tool_call_str.split("(")[1][:-1]

                if tool_name in self.tools:
                    tool_function = self.tools[tool_name]
                    result = tool_function(tool_arg)
                    final_answer = json.dumps(result, indent=4, ensure_ascii=False)
                else:
                    final_answer = f"오류: '{tool_name}'은(는) 알 수 없는 도구입니다."
            elif step.startswith("LLM_CALL:"):
                # LLM 호출
                prompt = step.replace("LLM_CALL:", "")
                messages = [{"role": "user", "content": prompt}]
                final_answer = self.llm_model.generate_response(messages)

        print("5. [Executor] 실행 완료.")
        return final_answer


## Agent 생성 및 실행

In [ ]:
# Agent 인스턴스 생성 (LLMModel 주입)
agent = Agent(llm_model)

# 사용자 쿼리 정의
query = "포토리소그래피 파라미터 정보 알려줘."

# 에이전트 실행
# 계획 수립 (Plan)
steps = agent.plan(query)

# 계획 실행 (Execute)
answer = agent.execute(steps)

# 결과 출력
print("\n\n--- 최종 답변 ---")
print(answer)

1. [Agent Init] 사용 가능한 툴: ['get_step_parameters']
1. [Agent Init] 에이전트 초기화 완료.
2. [Planner] 쿼리 분석 및 계획 수립 시작: '포토리소그래피 파라미터 정보 알려줘.'
3. [Planner] 계획 수립 완료: ['TOOL_CALL:get_step_parameters(포토리소그래피)']
4. [Executor] 계획 실행 시작...
--- [Tool Called] get_step_parameters(step_name='포토리소그래피') ---
5. [Executor] 실행 완료.


--- 최종 답변 ---
{
    "노광시간(Exposure Time)": "5.2초",
    "레진두께(PR Thickness)": "210nm",
    "초점(Focus)": "+0.1um"
}
